# LangChain para aplicações com LLMs — guia didático

Este notebook reúne, em ordem de aprendizado, os 15 vídeos (5 aulas × 3 vídeos) do curso de LangChain.

| Aula | Tema | Você vai construir |
|---|---|---|
| 1 | Fundamentos | prompt → modelo → parser, ferramentas (*tools*) e agentes com busca na web |
| 2 | LCEL avançado | composição com `|`, `invoke`/`stream`/`batch`, roteamento e *fallback* |
| 3 | RAG e memória | chunking, embeddings, busca vetorial (FAISS), multi-query e histórico em SQLite |
| 4 | Ferramentas seguras e API | schemas rigorosos (Pydantic), agente com `create_agent`, API FastAPI com autenticação |
| 5 | LangGraph | grafos de estado, ciclos, *human-in-the-loop*, multi-agentes e paralelismo |

## Como usar

1. **Chaves de API (opcional).** Muitas células chamam modelos reais (Google Gemini, OpenAI, Tavily). Crie um arquivo `.env` ao lado deste notebook (veja o `.example.env`) com suas chaves. No Google Colab, use os *Secrets*.
2. **Funciona sem chaves.** Sem chave, as células que dependem de um modelo usam um **modelo simulado** (respostas fixas) ou são **puladas com um aviso**. Assim você consegue executar tudo e entender a *estrutura* de cada exemplo — a resposta, porém, só é "inteligente" com um modelo real.
3. **Instalação por aula.** Cada aula começa com `%pip install` apenas das bibliotecas que ela usa.
4. Execute de cima para baixo.

> **Atenção a custos e limites:** chamadas a modelos reais podem ter custo e cotas. Os exemplos são pequenos de propósito.

---
# Preparação do ambiente

In [ ]:
# Instala o carregador de variáveis de ambiente e o pacote base do LangChain
%pip install -q python-dotenv langchain-core

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

# 1) Carrega as chaves do arquivo .env (procura a partir da pasta atual e sobe nas pastas)
load_dotenv(find_dotenv(usecwd=True))

# 2) No Google Colab, as chaves podem vir dos "Secrets" (ícone de chave na barra lateral)
try:
    from google.colab import userdata
    for nome in ("GOOGLE_API_KEY", "OPENAI_API_KEY", "TAVILY_API_KEY",
                 "API_KEY_ANALISTA", "API_KEY_DIRETORIA",
                 "GEMINI_CHAT_MODEL", "OPENAI_CHAT_MODEL"):
        if not os.getenv(nome):
            try:
                os.environ[nome] = userdata.get(nome)
            except Exception:
                pass          # segredo não cadastrado: segue sem ele
except ImportError:
    pass                      # não está no Colab

# 3) Flags: dizem quais serviços estão disponíveis neste ambiente
TEM_GOOGLE = bool(os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"))
TEM_OPENAI = bool(os.getenv("OPENAI_API_KEY"))
TEM_TAVILY = bool(os.getenv("TAVILY_API_KEY"))

# 4) Nomes dos modelos (podem ser trocados no .env se algum for descontinuado)
MODELO_GEMINI = os.getenv("GEMINI_CHAT_MODEL") or "gemini-2.5-flash"
MODELO_OPENAI = os.getenv("OPENAI_CHAT_MODEL") or "gpt-4o-mini"

print(f"Google/Gemini : {'chave encontrada' if TEM_GOOGLE else 'sem chave (modo simulado)'}")
print(f"OpenAI        : {'chave encontrada' if TEM_OPENAI else 'sem chave (modo simulado)'}")
print(f"Tavily (busca): {'chave encontrada' if TEM_TAVILY else 'sem chave'}")

### Modelo simulado

Para os exemplos rodarem **sem chave de API**, usamos o `FakeListChatModel`: um "modelo de brinquedo" que devolve respostas pré-definidas, em sequência. Ele se comporta como um modelo de chat de verdade dentro de uma *chain*, o que permite testar a estrutura sem custo.

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

def llm_simulado(respostas):
    """Cria um modelo de chat que devolve as `respostas` em ordem (e recomeça ao acabar)."""
    return FakeListChatModel(responses=respostas)

# Teste rápido: o "modelo" responde sempre com o próximo texto da lista
teste = llm_simulado(["Olá! Sou um modelo simulado."])
print(teste.invoke("Oi?").content)

---
# Aula 1 — Fundamentos: prompts, chains, ferramentas e agentes

O LangChain organiza aplicações com LLMs em **blocos reutilizáveis** que se encaixam:

- **Prompt template** — molde de mensagem com *variáveis*;
- **Modelo (LLM)** — gera a resposta;
- **Output parser** — transforma a resposta no formato que você quer (ex.: texto puro);
- **Chain** — a sequência desses blocos, encadeada com o operador `|` (**LCEL**, *LangChain Expression Language*).

In [ ]:
# Instala o que a Aula 1 usa: Google Gemini, agentes clássicos e busca Tavily
%pip install -q langchain-google-genai langchain-classic langchain-community tavily-python

## 1.1 Primeira chain: prompt → modelo → parser

Todos os blocos do LangChain são **Runnables**: têm os mesmos métodos (`invoke`, `stream`, `batch`) e podem ser ligados com `|`. A chain abaixo faz: *preencher o prompt → chamar o modelo → extrair o texto*.

Primeiro, só o **prompt template** (não precisa de chave): veja como o molde vira mensagens.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Molde com duas mensagens: instruções do sistema + pergunta do usuário com a variável {topico}
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente técnico especialista em arquitetura de software de Inteligência Artificial. Seja direto e profissional."),
    ("user", "Explique brevemente o conceito de {topico} e por que ele é modular."),
])

# invoke() preenche as variáveis e devolve as mensagens prontas para enviar ao modelo
mensagens = prompt_template.invoke({"topico": "LangChain Expression Language (LCEL)"})
for m in mensagens.to_messages():
    print(f"[{m.type}] {m.content}")

Agora montamos a chain completa. Com chave do Google usamos o **Gemini**; sem chave, o modelo simulado (a resposta será fixa, mas a estrutura é idêntica).

In [ ]:
parser = StrOutputParser()   # extrai só o texto (.content) da resposta do modelo

if TEM_GOOGLE:
    from langchain_google_genai import ChatGoogleGenerativeAI
    modelo = ChatGoogleGenerativeAI(model=MODELO_GEMINI, temperature=0)   # temperature=0: respostas estáveis
else:
    modelo = llm_simulado(["LCEL é uma forma declarativa de encadear componentes com '|'. É modular porque cada bloco tem a mesma interface e pode ser trocado."])

chain = prompt_template | modelo | parser     # a chain: prompt -> modelo -> parser

try:
    resposta = chain.invoke({"topico": "LangChain Expression Language (LCEL)"})
    print("=== RESPOSTA ===")
    print(resposta)
except Exception as erro:
    print("Falha ao chamar o modelo (confira chave, cota e nome do modelo):", type(erro).__name__, erro)

**O que observar:** trocar o modelo (Gemini, OpenAI, simulado...) **não muda** o resto da chain. É essa modularidade que o LCEL oferece.

## 1.2 Ferramentas (*tools*)

Um LLM só gera texto. Para **agir** (calcular, consultar um banco, buscar na web), damos a ele **ferramentas**: funções Python que ele pode pedir para executar. O decorador `@tool` transforma uma função em ferramenta; a **docstring** e os **tipos** viram a descrição que o modelo lê para decidir quando usá-la.

In [ ]:
from langchain_core.tools import tool

@tool
def calcular_custo_tokens(quantidade_tokens: int, modelo: str) -> str:
    """Calcula o custo estimado em dólares com base na quantidade de tokens e no modelo especificado."""
    modelo = modelo.lower()
    if "gpt-4o-mini" in modelo:
        custo = (quantidade_tokens / 1_000_000) * 0.15     # preço hipotético por milhão de tokens
        return f"O custo estimado para {quantidade_tokens} tokens no gpt-4o-mini é de ${custo:.6f} USD."
    return "Modelo não mapeado para cálculo de custo local."

# O que o modelo "enxerga" sobre a ferramenta: nome, descrição e argumentos
print("Nome       :", calcular_custo_tokens.name)
print("Descrição  :", calcular_custo_tokens.description)
print("Argumentos :", calcular_custo_tokens.args)

# Uma ferramenta também pode ser chamada diretamente, como qualquer Runnable
print("\nTeste direto:", calcular_custo_tokens.invoke({"quantidade_tokens": 5_000_000, "modelo": "gpt-4o-mini"}))

## 1.3 Agente com memória e ferramentas

Um **agente** é um LLM que, em *loop*, decide: "preciso usar uma ferramenta?" → executa → lê o resultado → decide de novo, até poder responder. Dois espaços especiais no prompt:

- `chat_history` — histórico da conversa (a "memória" de curto prazo);
- `agent_scratchpad` — rascunho onde o agente registra as ações e observações do loop.

> Estamos usando `AgentExecutor` (do pacote `langchain-classic`). Na Aula 4 você verá a forma moderna, `create_agent`. **Requer chave do Google.**

In [ ]:
import logging

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import MessagesPlaceholder

logging.getLogger("langchain_google_genai").setLevel(logging.ERROR)   # silencia avisos de schema

if TEM_GOOGLE:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(model=MODELO_GEMINI, temperature=0)
    tools = [calcular_custo_tokens]                      # ferramentas disponíveis

    prompt_agente = ChatPromptTemplate.from_messages([
        ("system", "Você é um agente financeiro de IA. Use as ferramentas disponíveis para responder dúvidas sobre custos se necessário."),
        MessagesPlaceholder(variable_name="chat_history"),        # histórico da conversa
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),    # rascunho do agente
    ])

    agente = create_tool_calling_agent(llm, tools, prompt_agente)
    executor = AgentExecutor(agent=agente, tools=tools, verbose=True)   # verbose mostra cada passo

    historico = [
        HumanMessage(content="Olá, meu nome é Carlos e estou construindo um assistente."),
        AIMessage(content="Olá Carlos! Como posso te ajudar com os custos ou arquitetura do seu assistente hoje?"),
    ]
    pergunta = "Se meu sistema rodar 5.000.000 de tokens no gpt-4o-mini, quanto vou gastar?"

    try:
        resultado = executor.invoke({"input": pergunta, "chat_history": historico})
        print("\n=== RESPOSTA FINAL ===")
        print(resultado["output"])
    except Exception as erro:
        print("Falha ao executar o agente:", type(erro).__name__, erro)
else:
    print("Célula pulada: precisa de GOOGLE_API_KEY. Com ela, o agente decide chamar a ferramenta "
          "`calcular_custo_tokens` e responde: US$ 0,75 para 5 milhões de tokens.")

**Como ler a saída (com `verbose=True`):** o agente mostra *Invoking: `calcular_custo_tokens` with {...}* (a decisão), o resultado da ferramenta (a observação) e, por fim, a resposta em linguagem natural.

## 1.4 Answer Engine: agente com busca na web

Modelos têm **data de corte** e não sabem fatos recentes. Dando uma ferramenta de busca em tempo real (**Tavily**), o agente consulta a internet antes de responder. **Requer chaves do Google e do Tavily** (cadastro gratuito em tavily.com).

In [ ]:
if TEM_GOOGLE and TEM_TAVILY:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
    from langchain_community.tools.tavily_search import TavilyAnswer
    from langchain_google_genai import ChatGoogleGenerativeAI

    busca_internet = TavilyAnswer()                       # ferramenta de busca com resposta resumida
    ferramentas = [busca_internet]

    llm_controladora = ChatGoogleGenerativeAI(model=MODELO_GEMINI, temperature=0)

    prompt_engine = ChatPromptTemplate.from_messages([
        ("system", ("Você é o 'Answer Engine', um motor de respostas integrado à internet. "
                    "Sempre que o usuário perguntar sobre fatos recentes, notícias ou cotações, "
                    "use a ferramenta de busca para trazer dados verídicos e cite as descobertas de forma resumida.")),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])

    agente_busca = create_tool_calling_agent(llm_controladora, ferramentas, prompt_engine)
    executor_busca = AgentExecutor(agent=agente_busca, tools=ferramentas, verbose=True,
                                   handle_parsing_errors=True)   # tolera respostas fora do formato

    try:
        resultado = executor_busca.invoke({"input": "Quem ganhou o último campeonato de Fórmula 1 e qual foi a pontuação final?"})
        print("\n=== RESPOSTA COM BUSCA EM TEMPO REAL ===")
        print(resultado["output"])
    except Exception as erro:
        print("Falha ao executar o Answer Engine:", type(erro).__name__, erro)
else:
    print("Célula pulada: precisa de GOOGLE_API_KEY e TAVILY_API_KEY.")

---
# Aula 2 — LCEL avançado: composição, roteamento e resiliência

In [ ]:
# Instala o que a Aula 2 usa: Pydantic (validação) e OpenAI (modelo alternativo para o fallback)
%pip install -q pydantic langchain-openai langchain-google-genai

## 2.1 Composição com `|`, `RunnableLambda` e `RunnablePassthrough`

Além de prompts e modelos, qualquer **função Python** pode virar parte da chain:

- `RunnableLambda(funcao)` — envolve uma função (ex.: limpeza de texto);
- `RunnablePassthrough()` — repassa o dado adiante sem alterar;
- um **dicionário** `{"chave": runnable}` dentro da chain monta o próximo *input*.

Na chain abaixo, o 1º modelo **gera** um texto, uma função **limpa**, e o 2º prompt pede a **revisão**.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Prompt 1: gera a descrição técnica
prompt_gerador = ChatPromptTemplate.from_template("Gere uma breve descrição técnica sobre o conceito de: {topico}")

# Prompt 2: revisa o texto gerado (a variável {texto_gerado} virá do passo anterior)
prompt_revisor = ChatPromptTemplate.from_template(
    "Melhore o seguinte texto técnico, corrigindo eventuais termos e tornando-o mais formal:\n\n{texto_gerado}"
)

def limpar_texto(texto: str) -> str:
    """Simula uma sanitização: tira espaços das pontas e espaços duplos."""
    return texto.strip().replace("  ", " ")

runnable_limpeza = RunnableLambda(limpar_texto)   # função comum -> bloco da chain

# Modelo real (Gemini) se houver chave; senão, simulado com 2 respostas (gerador e revisor)
if TEM_GOOGLE:
    from langchain_google_genai import ChatGoogleGenerativeAI
    modelo_lcel = ChatGoogleGenerativeAI(model=MODELO_GEMINI, temperature=0)
else:
    modelo_lcel = llm_simulado(["  Descrição técnica inicial (gerada pelo 1º passo).  ",
                                "Texto revisado e mais formal (gerado pelo 2º passo)."])

chain_lcel = (
    prompt_gerador
    | modelo_lcel
    | StrOutputParser()
    | runnable_limpeza
    | {"texto_gerado": RunnablePassthrough()}   # guarda o texto no dicionário esperado pelo próximo prompt
    | prompt_revisor
    | modelo_lcel
    | StrOutputParser()
)
print("Chain montada com sucesso.")

### Os três modos de execução

> No **modo simulado**, o `batch` roda as duas entradas em paralelo compartilhando a mesma lista de respostas fixas; por isso o texto de cada resultado pode sair trocado ou só com o 1º passo. Com um modelo real isso não acontece.

Toda chain LCEL oferece, sem código extra:

- `invoke(entrada)` — uma entrada, uma resposta;
- `stream(entrada)` — devolve a resposta **aos poucos** (ótimo para chats);
- `batch([entradas])` — várias entradas **em paralelo**.

In [ ]:
try:
    print("=== 1. invoke() ===")
    print(chain_lcel.invoke({"topico": "LangChain Expression Language (LCEL)"}))

    print("\n=== 2. stream() — pedaço por pedaço ===")
    for pedaco in chain_lcel.stream({"topico": "RunnableParallel"}):
        print(pedaco, end="", flush=True)    # flush=True mostra o texto assim que chega
    print()

    print("\n=== 3. batch() — duas entradas de uma vez ===")
    entradas = [{"topico": "Large Language Models"}, {"topico": "API Idempotency"}]
    for i, resultado in enumerate(chain_lcel.batch(entradas), 1):
        print(f"Resultado {i}: {resultado}")
except Exception as erro:
    print("Falha ao executar a chain:", type(erro).__name__, erro)

## 2.2 Roteamento com saída estruturada (Pydantic)

Às vezes a chain precisa **decidir um caminho**: pergunta financeira → especialista financeiro; pergunta técnica → especialista técnico. Para o modelo decidir de forma **confiável**, pedimos uma **saída estruturada**: em vez de texto livre, ele devolve um objeto validado por um **schema Pydantic**.

O validador abaixo normaliza a resposta (ignora maiúsculas e acentos) e **recusa** setores inválidos. Veja funcionar, sem precisar de modelo:

In [ ]:
import unicodedata

from pydantic import BaseModel, Field, ValidationError, field_validator

class RoteadorSuporte(BaseModel):
    """Classifica a intenção do usuário para direcionar ao setor correto."""

    setor: str = Field(description="O setor para onde a dúvida deve ser enviada baseado no contexto.")

    @field_validator("setor", mode="before")
    def _normalizar_setor(cls, v):
        # Antes de validar: tira acentos, espaços e maiúsculas; mapeia variações para o valor padrão
        if isinstance(v, str):
            s = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii").strip().lower()
            if "finance" in s:
                return "financeiro"
            if "tecn" in s or "suporte" in s:
                return "tecnico"
            return s
        return v

    @field_validator("setor")
    def _validar_setor(cls, v):
        # Depois de normalizar: só aceita os setores conhecidos
        if v not in ("financeiro", "tecnico"):
            raise ValueError("setor deve ser 'financeiro' ou 'tecnico'")
        return v

# Variações aceitas (são normalizadas)
for entrada in ["Financeiro", "FINANCEIRO ", "Técnico", "suporte"]:
    print(f"{entrada!r:15} -> {RoteadorSuporte(setor=entrada).setor}")

# Valor inválido: o Pydantic recusa
try:
    RoteadorSuporte(setor="marketing")
except ValidationError as erro:
    print("\nRejeitado:", erro.errors()[0]["msg"])

Agora o roteador completo. A chain faz duas coisas **em paralelo** (por ser um dicionário): classifica a intenção e repassa a pergunta original; em seguida uma função (`rotear_consulta`) escolhe a sub-chain especialista.

Sem chave, usamos um **classificador simulado** (regra de palavras-chave) no lugar do modelo.

In [ ]:
# Sub-chains especializadas: cada uma com seu próprio prompt
prompt_financeiro = ChatPromptTemplate.from_template(
    "Você é um especialista financeiro. Responda a seguinte dúvida sobre faturamento de forma clara e cordial: {pergunta}")
prompt_tecnico = ChatPromptTemplate.from_template(
    "Você é um engenheiro de software sênior. Responda a seguinte dúvida de TI de forma técnica e objetiva: {pergunta}")

if TEM_GOOGLE:
    from langchain_google_genai import ChatGoogleGenerativeAI
    modelo_rota = ChatGoogleGenerativeAI(model=MODELO_GEMINI, temperature=0)
    llm_classificador = modelo_rota.with_structured_output(RoteadorSuporte)   # devolve um objeto RoteadorSuporte
else:
    modelo_rota = llm_simulado(["(resposta simulada do especialista)"])

    def classificador_simulado(prompt_value):
        """Stand-in do modelo: decide o setor por palavras-chave."""
        texto = prompt_value.to_string().lower()
        palavras_fin = ("fatura", "estorno", "cobrança", "pagamento", "boleto")
        return RoteadorSuporte(setor="financeiro" if any(p in texto for p in palavras_fin) else "tecnico")

    llm_classificador = RunnableLambda(classificador_simulado)

chain_financeira = prompt_financeiro | modelo_rota | StrOutputParser()
chain_tecnica = prompt_tecnico | modelo_rota | StrOutputParser()

def rotear_consulta(dados):
    """Escolhe a sub-chain conforme a intenção detectada."""
    intencao = dados["intencao"].setor
    print(f"[Roteador] intenção detectada: {intencao.upper()}")
    chain_escolhida = chain_financeira if intencao == "financeiro" else chain_tecnica
    return chain_escolhida.invoke({"pergunta": dados["pergunta"]})

# O dicionário roda os dois itens e entrega o resultado à função de roteamento
chain_principal = {
    "intencao": ChatPromptTemplate.from_template("{pergunta}") | llm_classificador,
    "pergunta": RunnableLambda(lambda dados: dados["pergunta"]),   # repassa só o texto da pergunta
} | RunnableLambda(rotear_consulta)

try:
    for pergunta in ["Minha fatura veio duplicada este mês, como solicito o estorno?",
                     "Estou recebendo erro 500 ao tentar conectar no banco PostgreSQL local."]:
        print("\nPergunta:", pergunta)
        print("Resposta :", chain_principal.invoke({"pergunta": pergunta}))
except Exception as erro:
    print("Falha ao executar o roteador:", type(erro).__name__, erro)

**O que observar:** a linha `[Roteador] intenção detectada` mostra a decisão. Em produção, a saída estruturada evita parsear texto livre — o modelo é *obrigado* a devolver um objeto válido.

## 2.3 Ferramenta customizada e *fallback* entre modelos

Provedores de LLM falham (cota, lentidão, indisponibilidade). **Fallback** significa: se o modelo principal falhar, tenta o secundário automaticamente — com `.with_fallbacks([...])`.

Primeiro, a ferramenta (a **docstring** é o que o agente lê para decidir usá-la):

In [ ]:
@tool
def calcular_faturamento_anual(empresa: str, receita_mensal: float) -> str:
    """Calcula a projeção de faturamento anual de uma empresa baseada na receita mensal."""
    faturamento = receita_mensal * 12
    return f"A projeção de faturamento anual para a empresa {empresa} é de R$ {faturamento:,.2f}"

print(calcular_faturamento_anual.invoke({"empresa": "AlfaTech", "receita_mensal": 25000}))

### Fallback na prática (demonstração sem chave)

Aqui o "modelo primário" **sempre falha** (simulando um timeout) e o fallback assume — o mesmo mecanismo que será usado com modelos reais logo abaixo:

In [ ]:
def primario_que_falha(_entrada):
    raise TimeoutError("Tempo esgotado (simulado)")      # simula o provedor principal fora do ar

primario = RunnableLambda(primario_que_falha)
secundario = RunnableLambda(lambda entrada: f"Resposta do modelo secundário para: {entrada!r}")

resiliente = primario.with_fallbacks([secundario])        # se o primário levantar erro, usa o secundário
print(resiliente.invoke("Qual a projeção anual?"))

### Fallback com modelos reais (OpenAI → Gemini)

O modelo principal (OpenAI) é criado com `timeout=0.001` **de propósito**, para falhar sempre; o `BaseCallbackHandler` mostra no console cada tentativa. Assim você *vê* o fallback acontecer. **Requer chaves da OpenAI e do Google.**

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

class LogFallbackHandler(BaseCallbackHandler):
    """Imprime o que acontece a cada chamada de modelo (início, erro e sucesso)."""

    def on_llm_start(self, serialized, prompts, **kwargs):
        print("\n[TENTATIVA] chamando um modelo...")

    def on_llm_error(self, error, **kwargs):
        print(f"[FALHA] {type(error).__name__}. Redirecionando para o modelo secundário...")

    def on_llm_end(self, response, **kwargs):
        print("[SUCESSO] o modelo respondeu.")

if TEM_OPENAI and TEM_GOOGLE:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_openai import ChatOpenAI

    llm_primario = ChatOpenAI(model=MODELO_OPENAI, temperature=0, timeout=0.001, max_retries=0)   # falha de propósito
    llm_secundario = ChatGoogleGenerativeAI(model=MODELO_GEMINI, temperature=0)

    prompt_fb = ChatPromptTemplate.from_messages([
        ("system", "Você é um assistente financeiro corporativo. Use as ferramentas fornecidas sempre que precisar de dados calculados."),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    ferramentas_fb = [calcular_faturamento_anual]

    # Um agente para cada modelo; o fallback é aplicado no nível do agente
    agente_primario = create_tool_calling_agent(llm_primario, ferramentas_fb, prompt_fb)
    agente_secundario = create_tool_calling_agent(llm_secundario, ferramentas_fb, prompt_fb)
    agente_resiliente = agente_primario.with_fallbacks([agente_secundario])

    executor_fb = AgentExecutor(agent=agente_resiliente, tools=ferramentas_fb, verbose=True)
    try:
        resposta = executor_fb.invoke(
            {"input": "A empresa AlfaTech fatura 25000 reais por mês. Qual a projeção anual deles?"},
            config={"callbacks": [LogFallbackHandler()]},
        )
        print("\n=== RESPOSTA FINAL ===")
        print(resposta["output"])
    except Exception as erro:
        print("Falha ao executar o agente:", type(erro).__name__, erro)
else:
    print("Célula pulada: precisa de OPENAI_API_KEY e GOOGLE_API_KEY.")

---

# Aula 3 — RAG e memória

**RAG** (*Retrieval-Augmented Generation*) resolve duas limitações dos LLMs: eles **não conhecem os seus documentos** e **inventam respostas** quando não sabem. A ideia:

1. **Indexar:** quebrar os documentos em pedaços (*chunks*), transformar cada um em um **embedding** (vetor numérico) e guardar num **banco vetorial**;
2. **Recuperar:** dada uma pergunta, buscar os chunks **mais parecidos** (busca semântica);
3. **Gerar:** pedir ao LLM que responda **usando apenas** os trechos recuperados.

A segunda metade da aula trata de **memória**: fazer a conversa lembrar do que já foi dito, guardando o histórico num banco de dados.

In [ ]:
# Instala o que a Aula 3 usa: divisor de texto, FAISS (banco vetorial local) e provedores de embeddings
%pip install -q langchain-text-splitters langchain-community faiss-cpu langchain-google-genai langchain-openai

## 3.1 Pipeline de ingestão: chunking

Documentos grandes não cabem inteiros no contexto do modelo e geram buscas imprecisas. Por isso os dividimos em **chunks**. Dois parâmetros:

- `chunk_size` — tamanho máximo de cada pedaço (em caracteres);
- `chunk_overlap` — **sobreposição** entre pedaços vizinhos, para uma frase cortada no meio não perder o contexto.

O `RecursiveCharacterTextSplitter` tenta cortar em parágrafos, depois linhas, frases e palavras, nessa ordem — respeitando o sentido sempre que possível. *(Este passo roda sem chave.)*

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Documento simulado: um relatório financeiro fictício
documento_corporativo = """
Relatório de Desempenho Financeiro - Indústrias ACME (Maio 2026).
O faturamento consolidado no último mês atingiu R$ 12.5 milhões, representando uma alta de 8% em relação a abril.
O principal motor de crescimento foi a divisão de Microcontroladores, que expandiu sua operação global em 15%.
As despesas operacionais (OPEX) totalizaram R$ 4.2 milhões, mantendo-se dentro do orçamento previsto.
A projeção para o próximo trimestre é manter a taxa de crescimento acima de dois dígitos.
"""

# Pedaços de até 150 caracteres, com 30 de sobreposição
divisor = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = divisor.split_text(documento_corporativo)

print(f"{len(chunks)} chunks gerados:")
for i, chunk in enumerate(chunks, 1):
    print(f"  Chunk {i} ({len(chunk)} caracteres): {chunk.strip()}")

**O que observar:** o fim de um chunk costuma se repetir no início do próximo (a sobreposição). Teste `chunk_size=300` e `chunk_overlap=0` para ver a diferença.

## 3.2 Embeddings e busca vetorial (FAISS)

Um **embedding** transforma texto em um vetor de números, de modo que textos com **significado parecido** fiquem **próximos**. Assim, "Quanto a empresa faturou?" fica perto de "O faturamento atingiu R$ 12,5 milhões", mesmo com palavras diferentes.

O **FAISS** (da Meta) é uma biblioteca de busca por similaridade que roda **localmente**, sem servidor.

A função abaixo escolhe o provedor de embeddings: **Google (Gemini)** primeiro, **OpenAI** como reserva. Ela faz uma chamada de teste (`"ping"`) para validar a chave e o nome do modelo **antes** de indexar, e mascara chaves em mensagens de erro para você não vazá-las sem querer.

In [ ]:
import re

def mascarar_segredos(texto):
    """Troca chaves de API que apareçam em mensagens de erro por <REDACTED>."""
    return re.sub(r"(key=|AIza|sk-)[A-Za-z0-9_\-]{6,}", r"\1<REDACTED>", str(texto))

def criar_embeddings():
    """Devolve um objeto de embeddings (Google ou OpenAI) ou None se nenhum estiver disponível."""
    if TEM_GOOGLE:
        from langchain_google_genai import GoogleGenerativeAIEmbeddings
        base = os.getenv("GOOGLE_EMBEDDING_MODEL", "gemini-embedding-001").removeprefix("models/")
        # Algumas versões do pacote exigem o prefixo "models/"; tentamos as duas formas
        for nome in (base, f"models/{base}"):
            try:
                emb = GoogleGenerativeAIEmbeddings(model=nome)
                emb.embed_query("ping")                       # chamada mínima: valida chave e modelo
                print("Embeddings do Google OK (modelo:", nome, ")")
                return emb
            except Exception as erro:
                print(f"Modelo '{nome}' indisponível -> {type(erro).__name__}: {mascarar_segredos(erro)}")
    if TEM_OPENAI:
        try:
            from langchain_openai import OpenAIEmbeddings
            emb = OpenAIEmbeddings(model="text-embedding-3-small")
            emb.embed_query("ping")
            print("Embeddings da OpenAI OK")
            return emb
        except Exception as erro:
            print("Falha nos embeddings da OpenAI:", mascarar_segredos(erro))
    return None

embeddings = criar_embeddings()
if embeddings is None:
    print("Sem provedor de embeddings: as células de busca vetorial abaixo serão puladas "
          "(configure GOOGLE_API_KEY ou OPENAI_API_KEY).")

In [ ]:
from langchain_community.vectorstores import FAISS

if embeddings is not None:
    # Indexa: cada chunk vira um vetor guardado no FAISS
    banco_vetorial = FAISS.from_texts(chunks, embeddings)

    # Busca semântica: a pergunta também vira vetor e procuramos o chunk mais próximo
    pergunta = "Quanto a empresa faturou no mês passado?"
    resultados = banco_vetorial.similarity_search(pergunta, k=1)   # k=1: o único mais parecido

    print("Pergunta:", pergunta)
    print("Trecho mais relevante ->", resultados[0].page_content.strip())
else:
    print("Célula pulada: sem provedor de embeddings.")

**Como ler:** a pergunta não repete as palavras do documento ("faturou" x "faturamento", "mês passado" x "último mês"), e mesmo assim o chunk certo é encontrado — é a **busca semântica** em ação.

## 3.3 RAG avançado: Multi-Query

Uma única pergunta pode não encontrar tudo. Ex.: *"Quem lidera a empresa e como foi o lucro?"* mistura **dois assuntos**. A técnica **Multi-Query** pede ao LLM que gere **variações** da pergunta; buscamos cada variação e juntamos os resultados (sem repetir). Isso aumenta o ***recall*** (a chance de achar todos os trechos relevantes).

Duas funções auxiliares fazem esse trabalho — vamos testá-las **sem modelo**, para entender a lógica:

In [ ]:
from langchain_core.documents import Document

def extrair_variacoes(texto, limite):
    """Transforma a resposta do LLM em lista de perguntas (remove numeração, marcadores e linhas vazias)."""
    variacoes = []
    for linha in texto.splitlines():
        limpa = re.sub(r"^\s*(\d+[\.\)]|[-*•])\s*", "", linha).strip()
        if limpa:
            variacoes.append(limpa)
    return variacoes[:limite]

def buscar_multi_query(banco, consultas, k):
    """Busca cada consulta e devolve a união dos documentos, sem repetidos (a ordem é preservada)."""
    unicos = {}
    for consulta in consultas:
        for doc in banco.similarity_search(consulta, k=k):
            unicos.setdefault(doc.page_content, doc)     # setdefault ignora repetidos
    return list(unicos.values())

# Teste 1: a resposta típica de um LLM tem numeração e linhas em branco
resposta_llm = "1. Quem é o diretor executivo da ACME?\n\n2) Qual foi o resultado financeiro da ACME?"
print("Variações extraídas:", extrair_variacoes(resposta_llm, limite=2))

# Teste 2: um banco de mentira, para mostrar a união sem repetição
class BancoFalso:
    def similarity_search(self, consulta, k=2):
        base = {"diretor": [Document(page_content="O diretor é Carlos Eduardo.")],
                "lucro": [Document(page_content="O faturamento foi de R$ 12.5 milhões."),
                          Document(page_content="O diretor é Carlos Eduardo.")]}
        return next((v for chave, v in base.items() if chave in consulta), [])[:k]

docs = buscar_multi_query(BancoFalso(), ["quem é o diretor?", "qual o lucro?"], k=2)
print("Documentos únicos:", [d.page_content for d in docs])

**Como ler:** o documento "O diretor é Carlos Eduardo." foi encontrado por duas consultas, mas aparece **uma só vez** no resultado.

Agora o fluxo completo de RAG: **gerar variações → recuperar → responder só com o contexto**. Precisa de um modelo de chat e de embeddings.

In [ ]:
def criar_llm_chat():
    """Devolve um modelo de chat real (Google ou OpenAI) ou None se não houver chave."""
    if TEM_GOOGLE:
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODELO_GEMINI, temperature=0)
    if TEM_OPENAI:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODELO_OPENAI, temperature=0)
    return None

llm_chat = criar_llm_chat()

In [ ]:
from langchain_core.prompts import PromptTemplate

if embeddings is not None and llm_chat is not None:
    try:
        # 1) Base de conhecimento: três fatos sobre a empresa, indexados no FAISS
        textos = [
            "O faturamento consolidado da ACME em Maio de 2026 foi de R$ 12.5 milhões.",
            "A divisão de Microcontroladores teve expansão de 15% na operação global.",
            "O Diretor Executivo da ACME em 2026 é o Engenheiro Carlos Eduardo.",
        ]
        banco_rag = FAISS.from_texts(textos, embeddings)

        pergunta_usuario = "Quem lidera a empresa ACME e como foi o lucro?"
        print("[Pergunta original]", pergunta_usuario)

        # 2) Query transformation: o LLM escreve variações da pergunta
        prompt_variacoes = PromptTemplate.from_template(
            "Você é um assistente técnico de IA. Gere {qtd} variações alternativas da seguinte pergunta "
            "para ajudar na busca em um banco de dados vetorial.\n"
            "Responda APENAS com as perguntas, uma por linha, sem numeração e sem comentários.\n"
            "Pergunta original: {pergunta}"
        )
        chain_variacoes = prompt_variacoes | llm_chat | StrOutputParser()
        texto_variacoes = chain_variacoes.invoke({"pergunta": pergunta_usuario, "qtd": 2})
        variacoes = extrair_variacoes(texto_variacoes, 2)
        print("\n[Variações geradas pelo LLM]")
        for v in variacoes:
            print(" -", v)

        # 3) Retrieval: busca a pergunta original E cada variação, juntando sem repetir
        consultas = [pergunta_usuario] + variacoes
        documentos = buscar_multi_query(banco_rag, consultas, k=2)
        contexto = "\n".join(d.page_content for d in documentos)
        print(f"\n[Retrieval] {len(documentos)} documento(s) único(s):\n{contexto}")

        # 4) Generation: responde usando APENAS o contexto recuperado
        prompt_rag = PromptTemplate.from_template(
            "Você é um assistente corporativo. Responda à pergunta do usuário utilizando APENAS o contexto fornecido abaixo.\n"
            "Se não souber a resposta com base no contexto, diga honestamente que não encontrou a informação.\n\n"
            "Contexto:\n{contexto}\n\nPergunta: {pergunta}\n\nResposta resumida e direta:"
        )
        resposta = (prompt_rag | llm_chat | StrOutputParser()).invoke({"contexto": contexto, "pergunta": pergunta_usuario})
        print("\n[Resposta final]\n", resposta)
    except Exception as erro:
        print("Falha no fluxo de RAG:", type(erro).__name__, mascarar_segredos(erro))
else:
    print("Célula pulada: precisa de embeddings e de um modelo de chat.")

**O que observar:** o contexto reúne fatos de **assuntos diferentes** (diretor e faturamento). E, como o prompt manda responder *apenas* com o contexto, o modelo deve admitir quando não achou — por exemplo, a base fala de **faturamento**, não de **lucro**. Isso reduz alucinações.

## 3.4 Memória persistente com SQLite

Por padrão, cada chamada ao LLM é **independente**: ele não lembra da anterior. Para uma conversa com memória, guardamos o **histórico** e o reenviamos a cada turno. O `RunnableWithMessageHistory` automatiza isso: ele **lê o histórico** da sessão antes de chamar o modelo e **grava** a nova troca depois.

Para o histórico **sobreviver** a reinícios do programa, guardamos num banco **SQLite** (um arquivo local). Precisamos implementar uma classe com 3 operações: ler (`messages`), gravar (`add_messages`) e apagar (`clear`).

> **Detalhe técnico:** abrimos e fechamos a conexão a cada operação, porque o LangChain pode usar *threads* e uma conexão SQLite não pode ser compartilhada entre threads.

In [ ]:
import sqlite3
import uuid
from contextlib import closing
from pathlib import Path

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from langchain_core.runnables.history import RunnableWithMessageHistory

CAMINHO_BANCO = Path("memoria_chat.db")          # arquivo criado na pasta atual do notebook
TABELA = "historico_conversas"
TIPOS = {"human": HumanMessage, "ai": AIMessage}  # só persistimos mensagens do usuário e do assistente

class SQLiteChatMessageHistory(BaseChatMessageHistory):
    """Histórico de conversa guardado em SQLite (uma conexão por operação)."""

    def __init__(self, session_id: str):
        self.session_id = session_id
        self._garantir_tabela()

    def _conectar(self):
        con = sqlite3.connect(CAMINHO_BANCO, timeout=5.0)
        con.execute("PRAGMA journal_mode=WAL")      # evita erro 'database is locked'
        return con

    def _garantir_tabela(self):
        # 'with con' = transação: confirma se deu certo, desfaz se der erro
        with closing(self._conectar()) as con, con:
            con.execute(f"""
                CREATE TABLE IF NOT EXISTS {TABELA} (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    role TEXT NOT NULL CHECK (role IN ('human', 'ai')),
                    content TEXT NOT NULL,
                    criado_em TEXT DEFAULT CURRENT_TIMESTAMP
                )""")
            con.execute(f"CREATE INDEX IF NOT EXISTS idx_sessao ON {TABELA} (session_id, id)")

    @property
    def messages(self) -> list[BaseMessage]:
        """Lê todas as mensagens da sessão, em ordem."""
        with closing(self._conectar()) as con:
            linhas = con.execute(
                f"SELECT role, content FROM {TABELA} WHERE session_id = ? ORDER BY id ASC",
                (self.session_id,)).fetchall()
        return [TIPOS[role](content=content) for role, content in linhas if role in TIPOS]

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Grava novas mensagens (o '?' evita SQL injection)."""
        linhas = [(self.session_id, m.type, m.content) for m in messages if m.type in TIPOS]
        if linhas:
            with closing(self._conectar()) as con, con:
                con.executemany(f"INSERT INTO {TABELA} (session_id, role, content) VALUES (?, ?, ?)", linhas)

    def clear(self) -> None:
        """Apaga o histórico desta sessão."""
        with closing(self._conectar()) as con, con:
            con.execute(f"DELETE FROM {TABELA} WHERE session_id = ?", (self.session_id,))

print("Classe de histórico definida. Banco:", CAMINHO_BANCO.resolve())

Agora usamos a memória numa conversa de dois turnos. No turno 2 perguntamos algo que **só dá para responder se o modelo lembrar** do turno 1. Com modelo real ele responde certo; com o modelo simulado a resposta é fixa, mas **o histórico é gravado do mesmo jeito** — e você poderá conferi-lo no banco.

In [ ]:
llm_memoria = criar_llm_chat() or llm_simulado([
    "Olá, Lucas! Prazer em ajudar com o seu projeto em Python.",
    "(simulado) Seu nome é Lucas e você usa Python.",
])

prompt_memoria = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente técnico de banco de dados. Seja breve e amigável."),
    MessagesPlaceholder(variable_name="history"),     # aqui entra o histórico lido do SQLite
    ("human", "{input}"),
])

chain_com_memoria = RunnableWithMessageHistory(
    prompt_memoria | llm_memoria,
    get_session_history=SQLiteChatMessageHistory,     # função que devolve o histórico de uma sessão
    input_messages_key="input",
    history_messages_key="history",
)

id_sessao = f"sessao_{uuid.uuid4().hex[:8]}"          # id novo a cada execução: conversa limpa
config_sessao = {"configurable": {"session_id": id_sessao}}

try:
    print("[Turno 1] Aluno: Olá! Meu nome é Lucas e estou desenvolvendo o projeto final em Python.")
    r1 = chain_com_memoria.invoke({"input": "Olá! Meu nome é Lucas e estou desenvolvendo o projeto final em Python."},
                                  config=config_sessao)
    print("Assistente:", r1.content)

    print("\n[Turno 2] Aluno: Qual é o meu nome e qual linguagem estou usando?")
    r2 = chain_com_memoria.invoke({"input": "Qual é o meu nome e qual linguagem estou usando?"},
                                  config=config_sessao)
    print("Assistente:", r2.content)
except Exception as erro:
    print("Falha na conversa:", type(erro).__name__, mascarar_segredos(erro))

In [ ]:
# Confere o que ficou gravado no banco SQLite para esta sessão
with closing(sqlite3.connect(CAMINHO_BANCO)) as con:
    linhas = con.execute(
        f"SELECT id, role, substr(content, 1, 70) FROM {TABELA} WHERE session_id = ? ORDER BY id",
        (id_sessao,)).fetchall()

print(f"{len(linhas)} mensagens gravadas na sessão {id_sessao}:")
for linha in linhas:
    print(" ", linha)

> Você pode ver o aviso *"RunnableWithMessageHistory is deprecated"*: o LangChain recomenda hoje a persistência do LangGraph (veja a Aula 5, com `SqliteSaver`). O exemplo continua funcionando e é ótimo para entender a ideia de histórico por sessão.

**Como ler:** cada troca gera duas linhas (`human` e `ai`). Como o histórico está no **arquivo**, se você reiniciar o kernel e usar o mesmo `session_id`, a conversa continua de onde parou. Dica: abra `memoria_chat.db` no VS Code (extensão SQLite) para ver a tabela.

---

# Aula 4 — Ferramentas seguras, agentes e API

Quando o LLM **escolhe os argumentos** de uma ferramenta, esses argumentos são **entrada não confiável**: o modelo os inventa a partir do texto do usuário. Se a ferramenta acessa um banco de dados, isso é uma porta de entrada para erros e ataques. Nesta aula:

1. **Schema rigoroso** (Pydantic) — rejeita entradas inválidas **antes** de executar a ferramenta;
2. **Agente moderno** com `create_agent` e ferramenta com schema;
3. **API FastAPI** com autenticação e **autorização dentro da ferramenta** (não no prompt!).

In [ ]:
# Instala o que a Aula 4 usa: LangChain 1.x, OpenAI, FastAPI e httpx (cliente de testes do FastAPI)
%pip install -q langchain langchain-openai fastapi httpx pydantic

## 4.1 Schema como contrato e como proteção

O schema cumpre **dois papéis**:

1. **Rejeita** entradas inválidas em tempo de execução (não é só documentação);
2. É o **contrato que o LLM enxerga**: ao ver os valores permitidos (ex.: só 4 filiais), ele alucina menos.

`Literal[...]` cria uma **lista de valores permitidos** (*allowlist*); `Field(ge=2020, le=2030)` limita o intervalo do ano.

In [ ]:
import json
from typing import Literal

from langchain_core.tools import tool
from pydantic import BaseModel, Field

MESES = Literal["Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
                "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"]
FILIAIS = Literal["Geral", "São Paulo", "Recife", "Curitiba"]

class ConsultarVendasSchema(BaseModel):
    """Esquema de entrada com validação rigorosa."""
    mes: MESES = Field(..., description="Mês de referência")
    ano: int = Field(..., ge=2020, le=2030, description="Ano com 4 dígitos")
    filial: FILIAIS = Field("Geral", description="Nome da filial para filtrar")

@tool("consultar_vendas_db", args_schema=ConsultarVendasSchema)
def consultar_vendas(mes: str, ano: int, filial: str = "Geral"):
    """Consulta o total de vendas de uma filial em um mês específico.

    Use quando o usuário pedir faturamento, vendas ou performance de um período.
    Não use para projeções futuras nem para despesas.
    """
    # Em produção, aqui entra uma query SEMPRE parametrizada (nunca concatenar texto no SQL):
    #   cursor.execute("SELECT SUM(valor) FROM vendas WHERE mes=%s AND ano=%s AND filial=%s", (mes, ano, filial))
    return f"Resultado da consulta para {filial} em {mes}/{ano}: R$ 1.250.000,00"

# O JSON Schema que o LLM recebe: veja os valores permitidos e os limites
print("=== O que o LLM enxerga ===")
print(json.dumps(consultar_vendas.args, ensure_ascii=False, indent=2))

In [ ]:
# Chamada válida: passa pelo schema e executa
print("=== Chamada válida ===")
print(consultar_vendas.invoke({"mes": "Março", "ano": 2026}))

# Chamadas inválidas: o schema barra ANTES da ferramenta executar
print("\n=== Chamadas rejeitadas pelo schema ===")
invalidas = [
    {"mes": "banana", "ano": 2026},                              # mês inexistente
    {"mes": "Março", "ano": "abcd"},                             # ano não numérico
    {"mes": "Março", "ano": 1800},                               # fora da faixa permitida
    {"mes": "Março", "ano": 2026, "filial": "'; DROP TABLE--"},  # tentativa de injeção: fora da allowlist
]
for argumentos in invalidas:
    try:
        print("ACEITOU (não deveria):", consultar_vendas.invoke(argumentos))
    except Exception as erro:
        print(f"rejeitado -> {argumentos}  ({type(erro).__name__})")

**O que observar:** as 4 chamadas inválidas são **barradas** — inclusive a tentativa de *SQL injection* (`'; DROP TABLE--`), por não estar na lista de filiais permitidas. O schema é a primeira linha de defesa.

## 4.2 Agente com `create_agent`

Na Aula 1 usamos o `AgentExecutor` clássico. No LangChain 1.x, a forma recomendada é `create_agent`: você passa o **modelo**, as **ferramentas** e o **prompt de sistema**, e ele gerencia o loop (e o rascunho) sozinho.

Para ver cada passo do loop usamos `stream(..., stream_mode="values")` — o equivalente moderno do antigo `verbose=True`.

No segundo caso a filial **"Manaus"** não existe na allowlist: o schema **rejeita**, o erro volta ao agente como resultado da ferramenta e ele **se corrige**. **Requer chave da OpenAI.**

In [ ]:
from langchain_core.messages import HumanMessage

INSTRUCAO_SISTEMA = ("Você é um assistente financeiro especialista. Use as ferramentas disponíveis para responder. "
                     "Se a informação não vier de uma ferramenta, diga que não sabe.")

def executar_e_mostrar(agente, pergunta):
    """Roda o agente e imprime cada passo: decisão, resultado da ferramenta e resposta final."""
    print(f"\n[Pergunta] {pergunta}\n")
    vistas = 0                                              # quantas mensagens já mostramos
    for estado in agente.stream({"messages": [HumanMessage(content=pergunta)]}, stream_mode="values"):
        mensagens = estado["messages"]
        for msg in mensagens[vistas:]:                      # só as novas
            if msg.type == "ai" and msg.tool_calls:
                for chamada in msg.tool_calls:
                    print(f"  [decisão] chamar {chamada['name']} com {chamada['args']}")
            elif msg.type == "tool":
                print(f"  [ferramenta] {msg.content}")
            elif msg.type == "ai":
                print(f"\n[Resposta] {msg.content}")
        vistas = len(mensagens)

if TEM_OPENAI:
    from langchain.agents import create_agent
    from langchain_openai import ChatOpenAI

    agente_vendas = create_agent(ChatOpenAI(model=MODELO_OPENAI), tools=[consultar_vendas],
                                 system_prompt=INSTRUCAO_SISTEMA)
    try:
        executar_e_mostrar(agente_vendas, "Quanto vendemos na filial São Paulo em Janeiro de 2026?")   # caso 1: cabe na ferramenta
        executar_e_mostrar(agente_vendas, "E quanto vendemos na filial de Manaus em Janeiro de 2026?")  # caso 2: filial inexistente
    except Exception as erro:
        print("Falha ao executar o agente:", type(erro).__name__, erro)
else:
    print("Célula pulada: precisa de OPENAI_API_KEY.")

## 4.3 API FastAPI: autenticação e autorização

Agora expomos o agente como uma **API web**. Três conceitos de segurança:

- **Autenticação** — *quem é você?* A chave enviada no cabeçalho `x-api-key` é trocada por uma identidade (`analista_sp` ou `diretoria`). Padrão **deny-by-default**: sem chave válida, resposta **401**.
- **Autorização** — *o que você pode ver?* O analista só vê as filiais "Geral" e "São Paulo"; a diretoria vê todas.
- **Onde checar a autorização:** **dentro da ferramenta**, comparando com o usuário da requisição. **Nunca** pelo prompt: o LLM pode ignorar qualquer instrução de permissão.

Para a ferramenta saber **quem perguntou** sem passar pelo prompt, usamos uma `ContextVar` (variável de contexto): a API grava a identidade nela e a ferramenta lê.

In [ ]:
from contextvars import ContextVar

# Guarda a identidade de quem fez a requisição (a ferramenta lê daqui)
usuario_atual: ContextVar[str] = ContextVar("usuario_atual")

# Em produção isto viria de um banco ou provedor de identidade; aqui é só para a aula
FILIAIS_POR_USUARIO = {
    "analista_sp": {"Geral", "São Paulo"},
    "diretoria": {"Geral", "São Paulo", "Recife", "Curitiba"},
}

@tool("consultar_vendas_db", args_schema=ConsultarVendasSchema)
def consultar_vendas_seguro(mes: str, ano: int, filial: str = "Geral"):
    """Consulta o total de vendas de uma filial em um mês específico.

    Use quando o usuário pedir faturamento, vendas ou performance de um período.
    """
    # Autorização: compara a filial pedida com as permitidas ao usuário DA REQUISIÇÃO
    permitidas = FILIAIS_POR_USUARIO.get(usuario_atual.get(""), set())
    if filial not in permitidas:
        # Devolvemos texto (e não erro) para o agente conseguir explicar a recusa ao usuário
        return f"Acesso negado: este usuário não pode consultar a filial {filial}."
    return f"Resultado da consulta para {filial} em {mes}/{ano}: R$ 1.250.000,00"

# Demonstração direta (sem LLM): o MESMO pedido tem resultados diferentes conforme o usuário
for usuario in ("analista_sp", "diretoria"):
    token = usuario_atual.set(usuario)                    # simula o usuário da requisição
    try:
        resultado = consultar_vendas_seguro.invoke({"mes": "Janeiro", "ano": 2026, "filial": "Recife"})
    finally:
        usuario_atual.reset(token)                        # limpa o contexto
    print(f"{usuario:12s} -> {resultado}")

**O que observar:** o analista recebe *Acesso negado* para a filial Recife; a diretoria recebe o valor. A decisão foi tomada **no código**, não por um prompt.

Agora a API. A função `criar_api` recebe o agente (ou `None`) e devolve o app FastAPI. Assim a API pode ser **testada mesmo sem chave da OpenAI** (endpoints de saúde e autenticação).

In [ ]:
import logging

from fastapi import Depends, FastAPI, HTTPException, Security
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import APIKeyHeader

logger = logging.getLogger("api-financeira")
DEBUG_ERROS = os.getenv("DEBUG_ERROS", "0").lower() in {"1", "true", "sim"}   # mostra detalhes de erro só em desenvolvimento

# Chaves de demonstração, caso você não tenha definido as suas no .env
os.environ.setdefault("API_KEY_ANALISTA", "chave-analista-demo")
os.environ.setdefault("API_KEY_DIRETORIA", "chave-diretoria-demo")

# Lê o cabeçalho x-api-key (auto_error=False: nós mesmos devolvemos o 401)
esquema_api_key = APIKeyHeader(name="x-api-key", auto_error=False)

def autenticar(x_api_key: str = Security(esquema_api_key)) -> str:
    """Troca a chave do cabeçalho por uma identidade. Sem chave válida: 401 (deny-by-default)."""
    usuario = {
        os.environ.get("API_KEY_ANALISTA", ""): "analista_sp",
        os.environ.get("API_KEY_DIRETORIA", ""): "diretoria",
    }.get(x_api_key)
    if not x_api_key or not usuario:
        raise HTTPException(status_code=401, detail="Credencial inválida.")
    return usuario

# Contrato da API: o que entra e o que sai (validado pelo Pydantic)
class PerguntaRequest(BaseModel):
    pergunta: str = Field(..., min_length=3, max_length=500)

class RespostaAnalise(BaseModel):
    resposta: str
    ferramentas_usadas: list[str]
    detalhe_chamadas: list[dict] | None = None     # só preenchido com DEBUG_ERROS=1

def criar_api(agente):
    """Monta o app FastAPI. `agente` pode ser None (sem chave): os endpoints de análise respondem 503."""
    app = FastAPI(title="API Financeira Agêntica", version="1.0.0")

    app.add_middleware(CORSMiddleware,
                       allow_origins=os.getenv("CORS_ORIGENS", "http://localhost:3000").split(","),
                       allow_methods=["POST"],
                       allow_headers=["x-api-key", "content-type"])

    @app.post("/analise-financeira", response_model=RespostaAnalise)
    async def analise_financeira(corpo: PerguntaRequest, usuario: str = Depends(autenticar)) -> RespostaAnalise:
        if agente is None:
            raise HTTPException(status_code=503, detail="Agente indisponível (configure OPENAI_API_KEY).")
        token = usuario_atual.set(usuario)           # grava QUEM está perguntando para a ferramenta ler
        try:
            estado = await agente.ainvoke({"messages": [HumanMessage(content=corpo.pergunta)]})
        except Exception as erro:
            logger.exception("Falha ao invocar o agente (usuário=%s)", usuario)
            detalhe = f"{type(erro).__name__}: {erro}" if DEBUG_ERROS else "Falha ao consultar o assistente."
            raise HTTPException(status_code=502, detail=detalhe)
        finally:
            usuario_atual.reset(token)

        mensagens = estado["messages"]
        chamadas = [c for m in mensagens if m.type == "ai" and m.tool_calls for c in m.tool_calls]
        return RespostaAnalise(
            resposta=mensagens[-1].content,
            ferramentas_usadas=[c["name"] for c in chamadas],
            detalhe_chamadas=([{"ferramenta": c["name"], "argumentos": c["args"]} for c in chamadas]
                              if DEBUG_ERROS else None),
        )

    @app.get("/health")
    async def health():
        return {"status": "ok"}

    return app

# Cria o agente só se houver chave da OpenAI
agente_api = None
if TEM_OPENAI:
    from langchain.agents import create_agent
    from langchain_openai import ChatOpenAI
    agente_api = create_agent(ChatOpenAI(model=MODELO_OPENAI), tools=[consultar_vendas_seguro],
                              system_prompt=INSTRUCAO_SISTEMA)

app = criar_api(agente_api)
print("API criada. Agente:", "ativo" if agente_api else "indisponível (sem OPENAI_API_KEY)")

### Testando a API dentro do notebook

Em vez de subir um servidor (`uvicorn`), usamos o `TestClient` do FastAPI, que chama a API **na memória**. Testamos: saúde, **sem credencial** (401), **credencial errada** (401) e **credencial correta**.

In [ ]:
from fastapi.testclient import TestClient

cliente = TestClient(app)
pergunta_teste = {"pergunta": "Quanto vendemos em Janeiro de 2026 na filial Recife?"}

# 1) Saúde: não exige autenticação
print("1) /health                 ->", cliente.get("/health").json())

# 2) Sem credencial -> 401
print("2) sem chave               ->", cliente.post("/analise-financeira", json=pergunta_teste).status_code)

# 3) Credencial errada -> 401
r = cliente.post("/analise-financeira", json=pergunta_teste, headers={"x-api-key": "chave-que-nao-existe"})
print("3) chave inválida          ->", r.status_code)

# 4) Corpo inválido (pergunta curta demais) com chave válida -> 422 (validação do Pydantic)
r = cliente.post("/analise-financeira", json={"pergunta": "oi"}, headers={"x-api-key": os.environ["API_KEY_ANALISTA"]})
print("4) pergunta muito curta    ->", r.status_code)

# 5) Chave de analista: 200 com agente ativo, 503 sem OPENAI_API_KEY
r = cliente.post("/analise-financeira", json=pergunta_teste, headers={"x-api-key": os.environ["API_KEY_ANALISTA"]})
print("5) analista (filial Recife) ->", r.status_code)
print("   resposta:", r.json())

**Como ler os códigos HTTP:** `200` = sucesso; `401` = não autenticado; `422` = corpo da requisição inválido; `502` = falha ao falar com o LLM; `503` = serviço indisponível. No teste 5, **com chave da OpenAI**, a resposta do analista deve ser **"Acesso negado"** para Recife; com a chave da diretoria, o valor da consulta.

> **Para rodar como servidor de verdade** (fora do notebook), salve o código em `main.py` e execute no terminal `uvicorn main:app --reload`. O arquivo `testar_api.http` (extensão *REST Client* do VS Code) tem as requisições prontas.

---
# Aula 5 — LangGraph: grafos de estado, ciclos e multi-agentes

Chains (LCEL) são **lineares**: entrada → passos → saída, sem decisões e sem memória do caminho. O **LangGraph** modela o fluxo como uma **máquina de estados**:

- **Estado** — um dicionário compartilhado que todos os nós leem e atualizam;
- **Nós** — funções (com ou sem LLM) que recebem o estado e devolvem **só o que mudou**;
- **Arestas** — ligações entre nós; podem ser **condicionais** (decididas lendo o estado), inclusive **voltando** para um nó anterior (**ciclos**);
- **Reducers** — regras de como combinar valores (ex.: `operator.add` **soma listas** em vez de sobrescrever).

In [ ]:
# Instala o que a Aula 5 usa: LangGraph e o checkpointer SQLite (persistência do estado)
%pip install -q langgraph langgraph-checkpoint-sqlite langchain-openai

In [ ]:
import operator
import time
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

def llm_aula5(respostas_simuladas):
    """OpenAI real se houver chave; senão, modelo simulado com as respostas dadas."""
    if TEM_OPENAI:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODELO_OPENAI, temperature=0)
    return llm_simulado(respostas_simuladas)

print("Modo:", "OpenAI real" if TEM_OPENAI else "modelo simulado (sem chave)")

## 5.1 Da chain linear à máquina de estados

**Versão 1 — LCEL:** entra texto, sai texto. Não sobra nenhum rastro das decisões intermediárias.

**Versão 2 — StateGraph:** o mesmo problema com 3 nós — `classificar` (LLM decide a categoria), `responder` (LLM usa a categoria) e `auditar` (Python puro, sem LLM) — e uma **trilha de auditoria** acumulada pelo *reducer*.

In [ ]:
CATEGORIAS = ("financeiro", "tecnico", "comercial")

# Estado do grafo: o "contrato de dados" que circula entre os nós
class EstadoAtendimento(TypedDict):
    pergunta: str
    categoria: str
    resposta: str
    trilha: Annotated[list[str], operator.add]   # reducer: cada nó SOMA itens à lista

def criar_app_atendimento(llm):
    def classificar(estado: EstadoAtendimento) -> dict:
        """Nó 1: decide a categoria. Devolve SÓ o que mudou."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", f"Classifique a pergunta em exatamente uma palavra, entre: {', '.join(CATEGORIAS)}. Responda só a palavra."),
            ("human", "{pergunta}"),
        ])
        bruto = (prompt | llm | StrOutputParser()).invoke({"pergunta": estado["pergunta"]}).strip().lower()
        categoria = bruto if bruto in CATEGORIAS else "comercial"     # LLM é probabilístico: valide a saída!
        return {"categoria": categoria, "trilha": [f"classificar -> {categoria}"]}

    def responder(estado: EstadoAtendimento) -> dict:
        """Nó 2: usa a categoria decidida pelo nó anterior (veio pelo estado)."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Você é um especialista da área {categoria}. Responda em uma frase."),
            ("human", "{pergunta}"),
        ])
        texto = (prompt | llm | StrOutputParser()).invoke({"categoria": estado["categoria"], "pergunta": estado["pergunta"]})
        return {"resposta": texto, "trilha": ["responder"]}

    def auditar(estado: EstadoAtendimento) -> dict:
        """Nó 3: nem todo nó precisa de LLM; aqui é Python puro."""
        return {"trilha": [f"auditar -> resposta com {len(estado['resposta'])} caracteres"]}

    grafo = StateGraph(EstadoAtendimento)
    grafo.add_node("classificar", classificar)
    grafo.add_node("responder", responder)
    grafo.add_node("auditar", auditar)
    grafo.add_edge(START, "classificar")           # começo -> classificar
    grafo.add_edge("classificar", "responder")
    grafo.add_edge("responder", "auditar")
    grafo.add_edge("auditar", END)                 # auditar -> fim
    return grafo.compile()                         # compila: valida o grafo e o deixa executável

pergunta = "Nosso faturamento caiu 8% no trimestre. O que devo olhar primeiro?"
# Respostas simuladas, na ordem das chamadas: LCEL, classificar, responder, (stream) classificar, responder
llm_a = llm_aula5(["Comece pela margem e pela inadimplência do trimestre.", "financeiro",
                   "Revise margem, inadimplência e mix de produtos.", "financeiro",
                   "Revise margem, inadimplência e mix de produtos."])

try:
    # --- Versão 1: LCEL (linear, sem estado) ---
    prompt_lcel = ChatPromptTemplate.from_messages([
        ("system", "Você é um atendente corporativo. Responda em uma frase."), ("human", "{pergunta}")])
    print("=== 1. LCEL: entra texto, sai texto ===")
    print((prompt_lcel | llm_a | StrOutputParser()).invoke({"pergunta": pergunta}))

    # --- Versão 2: StateGraph ---
    app_atendimento = criar_app_atendimento(llm_a)
    print("\n=== 2. Desenho do grafo (formato Mermaid) ===")
    print(app_atendimento.get_graph().draw_mermaid())
    print("(cole o texto acima em https://mermaid.live para ver o diagrama)")

    print("\n=== 3. StateGraph: entra estado, sai estado ===")
    final = app_atendimento.invoke({"pergunta": pergunta, "categoria": "", "resposta": "", "trilha": []})
    print("Categoria:", final["categoria"])
    print("Resposta :", final["resposta"])
    print("Trilha acumulada pelo reducer:")
    for passo in final["trilha"]:
        print("  -", passo)

    print("\n=== 4. O que cada nó devolveu (stream_mode='updates') ===")
    for parcial in app_atendimento.stream({"pergunta": pergunta, "categoria": "", "resposta": "", "trilha": []},
                                          stream_mode="updates"):
        for no, mudanca in parcial.items():
            print(f"  [{no}] devolveu as chaves: {list(mudanca.keys())}")
except Exception as erro:
    print("Falha ao executar o grafo:", type(erro).__name__, erro)

**O que observar:**
- A `trilha` guarda **tudo que aconteceu**, na ordem — auditoria que a chain linear não oferece;
- cada nó devolve **apenas as chaves que mudou**; o LangGraph faz o *merge* no estado;
- o `stream` revela a decisão de cada nó em tempo real.

## 5.2 Ciclos, persistência e *human-in-the-loop*

Agora um grafo com **ciclo**: `gerar` escreve um resumo → `revisar` (o agente critica o próprio trabalho) → se reprovado, **volta** para `gerar`; se aprovado, vai para `publicar`.

Três recursos importantes:

- **Aresta condicional** (`decidir`) com **teto de tentativas** — sem ele um revisor exigente giraria para sempre, queimando tokens;
- **Checkpointer (SQLite)** — grava um *snapshot* do estado a cada passo; permite pausar, retomar e até "voltar no tempo";
- **`interrupt_before=["publicar"]`** — o grafo **pausa antes** de publicar e devolve o controle a um humano (*human-in-the-loop*): ele pode **editar o estado** e só então liberar.

In [ ]:
import uuid
from pathlib import Path

from langgraph.checkpoint.sqlite import SqliteSaver

CAMINHO_GRAFO_DB = Path("memoria_grafo_aula5.db")   # arquivo criado na pasta do notebook
MAX_TENTATIVAS = 3

class EstadoRedacao(TypedDict):
    tema: str
    rascunho: str
    critica: str
    aprovado: bool
    tentativas: int
    trilha: Annotated[list[str], operator.add]

def criar_app_redacao(llm, checkpointer):
    def gerar(estado: EstadoRedacao) -> dict:
        """Escreve (ou reescreve, se já houve crítica)."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Escreva UMA frase de resumo executivo sobre o tema. Se houver crítica anterior, corrija exatamente o que ela apontou."),
            ("human", "Tema: {tema}\nCrítica anterior: {critica}"),
        ])
        texto = (prompt | llm | StrOutputParser()).invoke(
            {"tema": estado["tema"], "critica": estado["critica"] or "(nenhuma, é a primeira versão)"})
        tentativa = estado["tentativas"] + 1
        return {"rascunho": texto, "tentativas": tentativa, "trilha": [f"gerar (tentativa {tentativa})"]}

    def revisar(estado: EstadoRedacao) -> dict:
        """O agente critica o próprio trabalho: é a autocrítica do ciclo."""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Você revisa resumos executivos. Se o texto tiver UMA frase e citar um número, responda exatamente APROVADO. "
                       "Caso contrário, responda com uma frase dizendo o que corrigir."),
            ("human", "{rascunho}"),
        ])
        parecer = (prompt | llm | StrOutputParser()).invoke({"rascunho": estado["rascunho"]}).strip()
        aprovado = parecer.upper().startswith("APROVADO")
        return {"critica": "" if aprovado else parecer, "aprovado": aprovado,
                "trilha": [f"revisar -> {'aprovado' if aprovado else 'reprovado'}"]}

    def publicar(estado: EstadoRedacao) -> dict:
        """Nó de alta criticidade: é aqui que o grafo vai pausar."""
        return {"trilha": [f"publicar -> {estado['rascunho'][:40]}..."]}

    def decidir(estado: EstadoRedacao) -> str:
        """Aresta condicional: calcula o próximo nó lendo o estado."""
        if estado["aprovado"]:
            return "publicar"
        if estado["tentativas"] >= MAX_TENTATIVAS:
            return "publicar"          # desiste de refinar e segue com o que tem
        return "gerar"                 # aresta que VOLTA: é o ciclo

    grafo = StateGraph(EstadoRedacao)
    grafo.add_node("gerar", gerar)
    grafo.add_node("revisar", revisar)
    grafo.add_node("publicar", publicar)
    grafo.add_edge(START, "gerar")
    grafo.add_edge("gerar", "revisar")
    grafo.add_conditional_edges("revisar", decidir, ["gerar", "publicar"])
    grafo.add_edge("publicar", END)
    # interrupt_before: para ANTES de 'publicar' e devolve o controle. Nada é publicado sem alguém olhar.
    return grafo.compile(checkpointer=checkpointer, interrupt_before=["publicar"])

print("Grafo de redação definido.")

Executamos em etapas: rodar até a pausa → inspecionar o estado → **editar** como humano → **aprovar** (retomar) → ver o histórico de *snapshots*.

> Cada execução usa um `thread_id` novo (a "conversa" do grafo). Reusar o mesmo id continuaria o estado salvo anteriormente.

In [ ]:
llm_r = llm_aula5(["A receita do trimestre caiu 8%, exigindo revisão imediata dos custos.", "APROVADO"])
config_grafo = {"configurable": {"thread_id": f"redacao_{uuid.uuid4().hex[:8]}"}}
entrada_redacao = {"tema": "queda de 8% no faturamento do trimestre", "rascunho": "", "critica": "",
                   "aprovado": False, "tentativas": 0, "trilha": []}

try:
    with SqliteSaver.from_conn_string(str(CAMINHO_GRAFO_DB)) as checkpointer:
        app_redacao = criar_app_redacao(llm_r, checkpointer)

        print("=== 1. Desenho do grafo (repare na aresta que volta para 'gerar') ===")
        print(app_redacao.get_graph().draw_mermaid())

        print("=== 2. Rodando até a pausa ===")
        for parcial in app_redacao.stream(entrada_redacao, config=config_grafo, stream_mode="updates"):
            for no, mudanca in parcial.items():
                if no == "__interrupt__":
                    print("  [PAUSA] interrompido antes de 'publicar'")
                    continue
                print(f"  [{no}] {mudanca['trilha'][0]}")

        print("\n=== 3. O grafo PAUSOU: estado nas mãos do humano ===")
        estado = app_redacao.get_state(config_grafo)
        print("  Próximo nó a executar:", estado.next)
        print("  Tentativas até aqui  :", estado.values["tentativas"])
        print("  Rascunho proposto    :", estado.values["rascunho"])

        print("\n=== 4. OVERRIDE: o humano edita o estado antes de liberar ===")
        app_redacao.update_state(config_grafo, {"rascunho": estado.values["rascunho"] + " (revisado por humano)"})
        print("  Rascunho após edição :", app_redacao.get_state(config_grafo).values["rascunho"])

        print("\n=== 5. APROVAÇÃO: invoke(None) retoma de onde parou ===")
        final = app_redacao.invoke(None, config=config_grafo)
        for passo in final["trilha"]:
            print("    -", passo)

        print("\n=== 6. TIME TRAVEL: snapshots salvos pelo checkpointer ===")
        historico = list(app_redacao.get_state_history(config_grafo))
        print(f"  {len(historico)} snapshots (do mais recente ao início):")
        for snap in historico[:6]:
            proximo = snap.next[0] if snap.next else "FIM"
            print(f"    próximo={proximo:<10} tentativas={snap.values.get('tentativas')}")
except Exception as erro:
    print("Falha ao executar o grafo de redação:", type(erro).__name__, erro)

**Como ler:**
- no passo 3, `next=('publicar',)` indica que o grafo está **parado antes** de publicar;
- no passo 4, o humano alterou o rascunho — a mudança fica **gravada no checkpoint**;
- no passo 5, `invoke(None)` significa "continue de onde parou", e o texto publicado já inclui a edição;
- no passo 6, cada snapshot tem um `checkpoint_id`: é possível **reexecutar a partir de qualquer ponto**.

## 5.3 Multi-agentes: o padrão Supervisor

Em vez de um agente que faz tudo, uma **equipe de especialistas** (pesquisador, analista, redator) coordenada por um **supervisor**. O supervisor **não executa**: só decide **quem trabalha agora** (ou `FINISH`). Cada especialista entrega e o controle **volta** ao supervisor.

Dois cuidados de engenharia:
1. **Valide** a decisão do supervisor (ele é um LLM) contra a lista real de especialistas;
2. Coloque um **limite de passos** (`MAX_PASSOS`) para evitar um supervisor indeciso girando para sempre.

In [ ]:
ESPECIALISTAS = ("pesquisador", "analista", "redator")
MAX_PASSOS = 6   # trava contra supervisor indeciso

class EstadoEquipe(TypedDict):
    tarefa: str
    proximo: str
    passos: int
    entregas: Annotated[list[str], operator.add]
    trilha: Annotated[list[str], operator.add]

def criar_app_supervisor(llm):
    def supervisor(estado: EstadoEquipe) -> dict:
        """Decide quem trabalha agora. Não executa nada: só roteia."""
        ja_feito = [e.split(":")[0] for e in estado["entregas"]]
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Você coordena uma equipe. Especialistas disponíveis: {equipe}.\nJá entregaram: {feito}.\n"
                       "Responda com UMA palavra: o próximo especialista, ou FINISH se a tarefa estiver completa. Só a palavra."),
            ("human", "Tarefa: {tarefa}"),
        ])
        bruto = (prompt | llm | StrOutputParser()).invoke({
            "equipe": ", ".join(ESPECIALISTAS),
            "feito": ", ".join(ja_feito) or "ninguém ainda",
            "tarefa": estado["tarefa"],
        }).strip().lower()
        proximo = bruto if bruto in ESPECIALISTAS else "FINISH"       # valida a decisão do LLM
        return {"proximo": proximo, "passos": estado["passos"] + 1, "trilha": [f"supervisor -> {proximo}"]}

    def criar_especialista(nome, instrucao):
        """Fábrica de nós: cada especialista tem a sua instrução."""
        def no(estado: EstadoEquipe) -> dict:
            prompt = ChatPromptTemplate.from_messages([("system", instrucao + " Responda em uma frase."),
                                                       ("human", "Tarefa: {tarefa}")])
            texto = (prompt | llm | StrOutputParser()).invoke({"tarefa": estado["tarefa"]})
            return {"entregas": [f"{nome}: {texto}"], "trilha": [nome]}
        return no

    def rotear(estado: EstadoEquipe) -> str:
        """Aresta condicional: lê a decisão do supervisor e devolve o próximo nó."""
        if estado["passos"] >= MAX_PASSOS:
            return END
        return estado["proximo"] if estado["proximo"] in ESPECIALISTAS else END

    grafo = StateGraph(EstadoEquipe)
    grafo.add_node("supervisor", supervisor)
    grafo.add_node("pesquisador", criar_especialista("pesquisador", "Você levanta dados e fontes."))
    grafo.add_node("analista", criar_especialista("analista", "Você interpreta números e aponta causas."))
    grafo.add_node("redator", criar_especialista("redator", "Você escreve o texto final para a diretoria."))
    grafo.add_edge(START, "supervisor")
    grafo.add_conditional_edges("supervisor", rotear, [*ESPECIALISTAS, END])
    for nome in ESPECIALISTAS:
        grafo.add_edge(nome, "supervisor")          # todo especialista devolve o controle ao supervisor
    return grafo.compile()

# Respostas simuladas na ordem das chamadas: supervisor, especialista, supervisor, ...
llm_eq = llm_aula5(["pesquisador", "Levantei os dados de vendas por região e canal.",
                    "analista", "A queda se concentra no Sul, por ruptura de estoque.",
                    "redator", "O faturamento caiu 8% pela ruptura de estoque no Sul; recomenda-se reposição imediata.",
                    "FINISH"])
try:
    app_equipe = criar_app_supervisor(llm_eq)
    print("=== O desenho da equipe ===")
    print(app_equipe.get_graph().draw_mermaid())

    final = app_equipe.invoke({"tarefa": "Explicar a queda de 8% no faturamento do trimestre",
                               "proximo": "", "passos": 0, "entregas": [], "trilha": []})
    print("=== Quem trabalhou, em ordem ===")
    for passo in final["trilha"]:
        print("  ", passo)
    print("\n=== O que cada especialista entregou ===")
    for entrega in final["entregas"]:
        print("  -", entrega)
except Exception as erro:
    print("Falha ao executar a equipe:", type(erro).__name__, erro)

**O que observar:** a trilha alterna `supervisor -> X` e o próprio especialista, até o supervisor devolver `FINISH`. Com modelo real, a ordem pode variar — por isso a validação e o limite de passos.

## 5.4 Latência: nós sequenciais x paralelos

Quando dois nós **não dependem um do outro** (ex.: consultar o CRM e o ERP), ligue os dois ao mesmo nó anterior: o LangGraph os executa **em paralelo** (*fan-out*). A única diferença é o **desenho das arestas**. Cada consulta aqui simula 1 segundo de espera:

In [ ]:
class EstadoConsulta(TypedDict):
    tarefa: str
    fontes: Annotated[list[str], operator.add]     # os dois nós adicionam à mesma lista

def consulta_lenta(nome, segundos=1.0):
    """Cria um nó que demora `segundos` (simula uma API ou um LLM lento)."""
    def no(estado: EstadoConsulta) -> dict:
        time.sleep(segundos)
        return {"fontes": [nome]}
    return no

def criar_app_sequencial():
    g = StateGraph(EstadoConsulta)
    g.add_node("crm", consulta_lenta("crm"))
    g.add_node("erp", consulta_lenta("erp"))
    g.add_edge(START, "crm")
    g.add_edge("crm", "erp")          # erp só começa quando o crm termina
    g.add_edge("erp", END)
    return g.compile()

def criar_app_paralelo():
    g = StateGraph(EstadoConsulta)
    g.add_node("crm", consulta_lenta("crm"))
    g.add_node("erp", consulta_lenta("erp"))
    g.add_edge(START, "crm")
    g.add_edge(START, "erp")          # duas arestas saindo do START = execução em paralelo
    g.add_edge("crm", END)
    g.add_edge("erp", END)
    return g.compile()

entrada_consulta = {"tarefa": "consolidar dados", "fontes": []}

inicio = time.perf_counter()
r_seq = criar_app_sequencial().invoke(entrada_consulta)
tempo_seq = time.perf_counter() - inicio

inicio = time.perf_counter()
r_par = criar_app_paralelo().invoke(entrada_consulta)
tempo_par = time.perf_counter() - inicio

print(f"sequencial (crm -> erp): {tempo_seq:.2f}s  fontes={r_seq['fontes']}")
print(f"paralelo   (crm | erp) : {tempo_par:.2f}s  fontes={sorted(r_par['fontes'])}")

**Como ler:** o tempo sequencial é a **soma** (~2 s) e o paralelo é o **maior** dos tempos (~1 s). Em produção, com chamadas a APIs e LLMs, isso reduz muito a latência.

---
# Resumo

| Conceito | Para que serve | Onde vimos |
|---|---|---|
| Prompt template | molde de mensagens com variáveis | 1.1 |
| LCEL (`\|`) | encadear blocos com a mesma interface (`invoke`/`stream`/`batch`) | 1.1, 2.1 |
| Tool | dar ao LLM a capacidade de agir | 1.2, 4.1 |
| Agente | LLM que decide, usa ferramentas e repete até concluir | 1.3, 4.2 |
| Saída estruturada | resposta validada por schema Pydantic | 2.2 |
| Fallback | trocar de modelo quando um falha | 2.3 |
| RAG | responder com base em documentos próprios | 3.1–3.3 |
| Memória | histórico persistente (SQLite) | 3.4 |
| Segurança de ferramentas | schema rigoroso + autorização no código | 4.1, 4.3 |
| LangGraph | estado, nós, arestas, ciclos, supervisor, paralelismo | 5.1–5.4 |

**Boas práticas que apareceram:** validar a saída do LLM, limitar ciclos, nunca confiar no prompt para segurança, registrar uma trilha de auditoria e pausar para aprovação humana em ações críticas.